# 🛡️ Phát hiện bình luận độc hại tiếng Việt — TF-IDF → BiLSTM → PhoBERT

Notebook này làm trọn các phần sau (đều dùng **số liệu thật do chính máy bạn chạy ra**):

| Phần | Nội dung |
|---|---|
| 1 | Đọc dữ liệu, xem phân bố nhãn |
| 2 | Làm sạch + tách từ tiếng Việt, và **khung so sánh TRƯỚC / SAU xử lý** |
| 3 | Chia train / val / test |
| 4 | Huấn luyện 3 mô hình: TF-IDF + Logistic Regression → BiLSTM → PhoBERT |
| 5 | **Bảng so sánh các mô hình** |
| 6 | **Ma trận nhầm lẫn** |
| 7 | **Phân tích lỗi**: câu đoán sai và lý do |
| 8 | **Nhập câu → ra kết quả** (như trang demo) |
| 9 | Mở rộng: lấy bình luận từ **YouTube API** và phân loại hàng loạt |

**Cách dùng nhanh (Google Colab):**
1. Bật GPU: *Runtime → Change runtime type → T4 GPU* (cần cho PhoBERT và BiLSTM).
2. Tải file dữ liệu `.csv` lên Colab, rồi sửa mục **CẤU HÌNH** ở ô code đầu tiên.
3. Chạy lần lượt từ trên xuống dưới.

> ⚠️ Mọi con số trong bảng và biểu đồ chỉ có ý nghĩa khi chạy trên **dữ liệu thật của nhóm**. Nếu bật `USE_DEMO_DATA = True`, đó chỉ là dữ liệu mẫu rất nhỏ để thử chạy, **không dùng để báo cáo**.

In [ ]:
# Cài thư viện (Colab đã có sẵn torch, pandas, scikit-learn...)
!pip install -q transformers pyvi seaborn

## ⚙️ Cấu hình (chỉ cần sửa ô này)

- Dữ liệu cần **một file CSV** có 1 cột chứa bình luận và 1 cột chứa nhãn.
- Ví dụ bộ **ViHSD** thường có cột `free_text` và `label_id` (0 = bình thường, 1 = xúc phạm, 2 = thù ghét). Nếu bạn có 3 file train/dev/test riêng, hãy ghép chúng lại thành 1 file bằng `pd.concat` rồi lưu CSV (notebook sẽ tự chia lại).
- `TOXIC_IDS` là danh sách số nhãn được tính là **độc hại**. Dùng để tính Precision/Recall của nhóm Toxic.

In [ ]:
import os, re, time, random, pickle, unicodedata, warnings, copy
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, confusion_matrix

warnings.filterwarnings("ignore")
SEED = 42
random.seed(SEED); np.random.seed(SEED)

# ===================== CẤU HÌNH (CHỈ SỬA Ở ĐÂY) =====================
DATA_PATH    = "data/raw/comments.csv"          # đường dẫn file CSV của bạn
TEXT_COL     = "free_text"                      # tên cột chứa bình luận
LABEL_COL    = "label_id"                       # tên cột chứa nhãn (số 0,1,2... hoặc tên nhãn)
LABEL_NAMES  = ["Bình thường", "Xúc phạm", "Thù ghét"]   # tên nhãn theo thứ tự 0,1,2
TOXIC_IDS    = [1, 2]                           # nhãn nào được coi là độc hại

USE_DEMO_DATA = False    # True = chạy thử bằng dữ liệu mẫu nhỏ (KHÔNG dùng để báo cáo)
RUN_LSTM      = True     # False nếu muốn bỏ qua BiLSTM
RUN_PHOBERT   = True     # False nếu chưa có GPU / muốn chạy nhanh

MAX_LEN        = 128               # độ dài tối đa của 1 bình luận (số từ / token)
LSTM_EPOCHS    = 6
PHOBERT_NAME   = "vinai/phobert-base"
PHOBERT_EPOCHS = 3
PHOBERT_BATCH  = 32                # nếu hết bộ nhớ GPU thì giảm xuống 16
# ======================================================================

for thu_muc in ["data/raw", "data/processed", "outputs"]:
    os.makedirs(thu_muc, exist_ok=True)
print("Đã sẵn sàng. Thư mục kết quả: outputs/ | Dữ liệu trung gian: data/processed/")

## 1️⃣ Đọc dữ liệu và xem phân bố nhãn

Bước này trả lời: *có bao nhiêu bình luận, mỗi nhóm chiếm bao nhiêu %?* Thường nhóm Toxic ít hơn nhiều → đây là lý do ta dùng **F1-macro** và **Recall** chứ không chỉ dùng Accuracy.

In [ ]:
def tao_du_lieu_mau(n_moi_lop=300):
    """Dữ liệu mẫu CỰC NHỎ chỉ để thử chạy notebook. KHÔNG phải dữ liệu thật."""
    sach = ["video này hay quá, cảm ơn bạn nhiều nha", "sản phẩm dùng tốt, giao hàng nhanh",
            "mình thấy shop tư vấn rất nhiệt tình", "bài giảng dễ hiểu, mong ra thêm video",
            "quán này đồ ăn ngon, giá hợp lý", "hôm nay trời đẹp quá mọi người ơi",
            "cho mình hỏi cách đặt hàng với", "ko ngờ dc giao sớm vậy, hài lòng lắm",
            "nội dung bổ ích, chúc kênh phát triển", "đội nhà đá hay thật sự, tự hào quá"]
    doc = ["mày ngu như bò, câm miệng đi", "đồ lừa đảo, cút khỏi đây",
           "thằng này nhìn là thấy rác rưởi", "bọn bán hàng như l**, đừng mua",
           "đm mày làm ăn kiểu gì vậy", "b.ắ.c k.ỳ toàn lũ ăn hại",
           "cái đồ vô học, biến đi cho khuất mắt", "sản phẩm như rác, shop lừa đảo ko dc tin",
           "ngu thì im đi, đừng có phát biểu", "lũ óc chó này chỉ giỏi chửi bậy"]
    them = ["", "haha", "thật sự", "mn ơi", "ad ơi", "😂", "https://abc.vn/xyz", "quááááá"]
    rows = []
    for nhan, goc in [(0, sach), (1, doc)]:
        for _ in range(n_moi_lop):
            rows.append((random.choice(them) + " " + random.choice(goc) + " " + random.choice(them), nhan))
    return pd.DataFrame(rows, columns=["text", "label"])

if USE_DEMO_DATA:
    TEXT_COL, LABEL_COL = "text", "label"
    LABEL_NAMES, TOXIC_IDS = ["Bình thường", "Toxic"], [1]
    df_goc = tao_du_lieu_mau()
    print("⚠️⚠️ ĐANG DÙNG DỮ LIỆU MẪU NHỎ — KẾT QUẢ CHỈ ĐỂ THỬ CHẠY, KHÔNG ĐƯA VÀO BÁO CÁO ⚠️⚠️")
else:
    if not os.path.exists(DATA_PATH):
        raise FileNotFoundError(f"Không thấy file '{DATA_PATH}'. Hãy tải file CSV lên và sửa DATA_PATH ở ô CẤU HÌNH.")
    df_goc = pd.read_csv(DATA_PATH)

df = df_goc[[TEXT_COL, LABEL_COL]].copy()
df.columns = ["text", "label"]
df = df.dropna().drop_duplicates(subset="text").copy()
df["text"] = df["text"].astype(str)

# Nếu nhãn là chữ (vd "CLEAN") thì đổi sang số theo thứ tự LABEL_NAMES
if df["label"].dtype == object:
    ten_sang_so = {ten: i for i, ten in enumerate(LABEL_NAMES)}
    df["label"] = df["label"].map(ten_sang_so)
    if df["label"].isna().any():
        raise ValueError("Có nhãn dạng chữ không khớp LABEL_NAMES. Hãy sửa LABEL_NAMES cho đúng tên nhãn trong file.")
df["label"] = df["label"].astype(int)
N_LOP = len(LABEL_NAMES)

df_goc_luu = df.copy()
df_goc_luu.to_csv("data/processed/01_du_lieu_goc.csv", index=False)

bang_nhan = df["label"].value_counts().sort_index().rename("Số bình luận").to_frame()
bang_nhan["Tên nhãn"] = [LABEL_NAMES[i] for i in bang_nhan.index]
bang_nhan["Tỷ lệ %"] = (bang_nhan["Số bình luận"] / len(df) * 100).round(2)
print(f"Tổng số bình luận sau khi bỏ trùng và bỏ dòng trống: {len(df):,}")
display(bang_nhan)

plt.figure(figsize=(6, 3.5))
sns.barplot(x=bang_nhan["Tên nhãn"], y=bang_nhan["Số bình luận"], palette="Set2")
plt.title("Phân bố nhãn trong dữ liệu"); plt.xlabel(""); plt.tight_layout()
plt.savefig("outputs/phan_bo_nhan.png", dpi=150); plt.show()

## 2️⃣ Làm sạch và tách từ tiếng Việt + khung so sánh TRƯỚC / SAU

Bình luận mạng xã hội rất "bẩn": có link, teencode (`ko`, `dc`, `mn`), kéo dài chữ (`quááááá`), chèn dấu chấm để né bộ lọc (`b.ắ.c k.ỳ`)...

Các bước làm sạch:
1. Chuẩn hóa Unicode (NFC) và đưa về chữ thường.
2. Bỏ thẻ HTML, link, thẻ `@tên`.
3. Ghép lại chữ bị chèn ký tự (`b.ắ.c` → `bắc`).
4. Rút gọn chữ kéo dài (`quááááá` → `quá`).
5. Đổi teencode phổ biến sang chữ đầy đủ (`ko` → `không`).
6. **Tách từ** bằng `pyvi` (ghép từ ghép bằng dấu `_`, ví dụ `lừa_đảo`). PhoBERT cần bước này.

> Từ điển teencode bên dưới chỉ là **bản khởi đầu**. Bạn nên bổ sung thêm từ xuất hiện nhiều trong dữ liệu của nhóm.

In [ ]:
from pyvi import ViTokenizer

TEENCODE = {
    "ko": "không", "k": "không", "kh": "không", "hok": "không", "khong": "không", "hông": "không",
    "dc": "được", "đc": "được", "dk": "được", "duoc": "được",
    "vs": "với", "j": "gì", "ntn": "như thế nào", "mn": "mọi người", "mk": "mình", "mik": "mình",
    "bn": "bao nhiêu", "bt": "bình thường", "đt": "điện thoại", "sp": "sản phẩm", "ae": "anh em",
    "ad": "admin", "cmt": "bình luận", "trc": "trước", "r": "rồi", "z": "vậy", "wa": "quá", "qa": "quá",
    "thik": "thích", "iu": "yêu", "bit": "biết", "ok": "ổn", "oke": "ổn", "dm": "đm", "dcm": "đcm",
    "đg": "đang", "lm": "làm", "ng": "người", "nc": "nói chuyện", "ib": "nhắn tin", "tks": "cảm ơn", "thanks": "cảm ơn",
}
RE_URL      = re.compile(r"(https?://\S+|www\.\S+|\bt\.me/\S+)", re.I)
RE_HTML     = re.compile(r"<[^>]+>")
RE_MENTION  = re.compile(r"@\w+")
RE_KEO_DAI  = re.compile(r"(\w)\1{2,}")
RE_CHEN_KT  = re.compile(r"\b\w(?:[\.\-_\*]\w)+\b")   # b.ắ.c  k.ỳ  đ.ị.t
RE_EMOJI    = re.compile(r"[\U0001F300-\U0001FAFF\u2600-\u27BF]")
RE_TU       = re.compile(r"\b(\w+)\b")

def phat_hien(raw):
    """Đếm các 'dấu hiệu bẩn' trong bình luận gốc (dùng cho khung TRƯỚC/SAU và phân tích lỗi)."""
    thap = unicodedata.normalize("NFC", raw).lower()
    return {
        "so_url": len(RE_URL.findall(thap)),
        "so_teencode": sum(1 for t in RE_TU.findall(thap) if t in TEENCODE),
        "so_keo_dai": len(RE_KEO_DAI.findall(thap)),
        "so_chen_ky_tu": len(RE_CHEN_KT.findall(thap)),
        "co_emoji": bool(RE_EMOJI.search(thap)),
    }

def lam_sach(text):
    t = unicodedata.normalize("NFC", str(text)).lower()
    t = RE_HTML.sub(" ", t)
    t = RE_URL.sub(" ", t)
    t = RE_MENTION.sub(" ", t)
    t = RE_CHEN_KT.sub(lambda m: re.sub(r"[\.\-_\*]", "", m.group()), t)
    t = RE_KEO_DAI.sub(r"\1", t)
    t = RE_TU.sub(lambda m: TEENCODE.get(m.group(1), m.group(1)), t)
    return re.sub(r"\s+", " ", t).strip()

def tach_tu(text_sach):
    return ViTokenizer.tokenize(text_sach)

t0 = time.time()
df_sach = df.copy()
df_sach["text_clean"] = df_sach["text"].apply(lam_sach)
df_sach["text_seg"] = df_sach["text_clean"].apply(tach_tu)
df_sach = df_sach[df_sach["text_seg"].str.len() > 0].copy().reset_index(drop=True)
df_sach.to_csv("data/processed/02_du_lieu_sach.csv", index=False)
print(f"Xong làm sạch + tách từ {len(df_sach):,} bình luận trong {time.time()-t0:.1f} giây. Đã lưu data/processed/02_du_lieu_sach.csv")

### 🔍 Khung so sánh TRƯỚC / SAU xử lý

Bảng dưới lấy vài câu ví dụ (có sẵn + vài câu ngẫu nhiên từ dữ liệu của bạn) để cho thấy văn bản thay đổi thế nào, và ghi rõ **phát hiện được gì**. Bạn có thể thêm câu của riêng mình vào `cau_vi_du`.

In [ ]:
cau_vi_du = [
    "hàng như l** ý mn đừng mua lừa đảo đấy k dùng dc đâu https://shopee.vn/fake-shop",
    "Bọn b.ắ.c k.ỳ ăn rau muống phá hoại văn hóa...",
    "Video bổ ích và chất lượng quááááá 😍 chúc kênh phát triển hơn nữa nhé!",
]
cau_ngau_nhien = df_sach["text"].sample(min(3, len(df_sach)), random_state=SEED).tolist()

def so_sanh_truoc_sau(danh_sach_cau):
    hang = []
    for raw in danh_sach_cau:
        sach = lam_sach(raw); seg = tach_tu(sach); dh = phat_hien(raw)
        ghi_chu = []
        if dh["so_url"]: ghi_chu.append(f"{dh['so_url']} link")
        if dh["so_teencode"]: ghi_chu.append(f"{dh['so_teencode']} teencode/viết tắt")
        if dh["so_keo_dai"]: ghi_chu.append("chữ kéo dài")
        if dh["so_chen_ky_tu"]: ghi_chu.append("chèn ký tự để né lọc")
        if dh["co_emoji"]: ghi_chu.append("có emoji")
        hang.append({"TRƯỚC (raw)": raw, "SAU làm sạch": sach, "SAU tách từ (vào mô hình)": seg,
                     "Đã phát hiện": ", ".join(ghi_chu) if ghi_chu else "—",
                     "Số ký tự trước → sau": f"{len(raw)} → {len(seg)}"})
    return pd.DataFrame(hang)

df_truoc_sau = so_sanh_truoc_sau(cau_vi_du + cau_ngau_nhien)
df_truoc_sau.to_csv("outputs/truoc_sau_xu_ly.csv", index=False)
pd.set_option("display.max_colwidth", 120)
display(df_truoc_sau)

# Thống kê trên toàn bộ dữ liệu: bao nhiêu % bình luận "bẩn" theo từng loại
dau_hieu = df_sach["text"].apply(phat_hien).apply(pd.Series)
thong_ke = pd.Series({
    "Có link": (dau_hieu["so_url"] > 0).mean() * 100,
    "Có teencode / viết tắt": (dau_hieu["so_teencode"] > 0).mean() * 100,
    "Chữ kéo dài": (dau_hieu["so_keo_dai"] > 0).mean() * 100,
    "Chèn ký tự né lọc": (dau_hieu["so_chen_ky_tu"] > 0).mean() * 100,
    "Có emoji": dau_hieu["co_emoji"].mean() * 100,
}).round(2)
print("Tỷ lệ % bình luận chứa từng dấu hiệu cần xử lý (toàn bộ dữ liệu):")
display(thong_ke.to_frame("Tỷ lệ %"))
plt.figure(figsize=(7, 3.5)); thong_ke.sort_values().plot(kind="barh", color="#4c78a8")
plt.xlabel("% bình luận"); plt.title("Mức độ 'bẩn' của dữ liệu thô"); plt.tight_layout()
plt.savefig("outputs/thong_ke_dau_hieu_ban.png", dpi=150); plt.show()

## 3️⃣ Chia dữ liệu train / val / test (80% / 10% / 10%)

- **train**: để mô hình học.
- **val**: để chọn epoch tốt nhất (không học trên đây).
- **test**: chỉ dùng **một lần ở cuối** để báo cáo kết quả, tránh "gian lận" vô tình.

Chia theo nhãn (`stratify`) để tỷ lệ các nhóm giống nhau ở cả 3 tập.

In [ ]:
train_val, test = train_test_split(df_sach, test_size=0.10, stratify=df_sach["label"], random_state=SEED)
train, val = train_test_split(train_val, test_size=1/9, stratify=train_val["label"], random_state=SEED)
train, val, test = [d.copy().reset_index(drop=True) for d in (train, val, test)]
train.to_csv("data/processed/03_train.csv", index=False)
val.to_csv("data/processed/03_val.csv", index=False)
test.to_csv("data/processed/03_test.csv", index=False)
print(f"train = {len(train):,} | val = {len(val):,} | test = {len(test):,}. Đã lưu 3 file CSV trong data/processed/")

y_train, y_val, y_test = train["label"].values, val["label"].values, test["label"].values
X_train, X_val, X_test = train["text_seg"].tolist(), val["text_seg"].tolist(), test["text_seg"].tolist()

## 4️⃣ Huấn luyện 3 mô hình

Cả 3 mô hình đều có cùng cách dùng: nhận **danh sách bình luận đã tách từ** → trả về **xác suất của từng nhãn**. Nhờ vậy ta so sánh công bằng và tái sử dụng cho ma trận nhầm lẫn, phân tích lỗi và demo.

Các chỉ số đo cho mỗi mô hình:
- **Accuracy**: tỷ lệ đoán đúng chung.
- **F1-macro**: trung bình F1 của các nhóm, không bị nhóm đông lấn át. **Đây là chỉ số chính.**
- **Toxic Precision**: trong những câu mô hình báo "độc hại", bao nhiêu % thật sự độc hại.
- **Toxic Recall**: trong những câu thật sự độc hại, mô hình bắt được bao nhiêu %.
- **Độ trễ**: thời gian xử lý 1 bình luận (ms). **Kích thước**: dung lượng mô hình (MB).

In [ ]:
KET_QUA = []        # mỗi mô hình 1 dòng chỉ số
DU_DOAN = {}        # tên mô hình -> ma trận xác suất trên tập test
HAM_DU_DOAN = {}    # tên mô hình -> hàm nhận list bình luận đã tách từ, trả xác suất

def do_do_tre(ham, texts, n=100):
    """Đo thời gian xử lý trung bình cho 1 bình luận (xử lý từng câu một, như khi chạy demo)."""
    mau = list(texts)[:n]
    ham(mau[:5])  # chạy thử cho nóng máy
    t0 = time.perf_counter()
    for t in mau:
        ham([t])
    return (time.perf_counter() - t0) / len(mau) * 1000

def ghi_ket_qua(ten, ham, giay_train, mb):
    proba = ham(X_test)
    y_pred = proba.argmax(1)
    y_that_nhi = np.isin(y_test, TOXIC_IDS).astype(int)
    y_doan_nhi = np.isin(y_pred, TOXIC_IDS).astype(int)
    hang = {
        "Mô hình": ten,
        "Accuracy": accuracy_score(y_test, y_pred),
        "F1-macro": f1_score(y_test, y_pred, average="macro"),
        "Toxic Precision": precision_score(y_that_nhi, y_doan_nhi, zero_division=0),
        "Toxic Recall": recall_score(y_that_nhi, y_doan_nhi, zero_division=0),
        "Độ trễ (ms/bình luận)": do_do_tre(ham, X_test),
        "Kích thước (MB)": mb,
        "Thời gian huấn luyện (giây)": giay_train,
    }
    KET_QUA.append(hang); DU_DOAN[ten] = proba; HAM_DU_DOAN[ten] = ham
    print(f"[{ten}] Accuracy = {hang['Accuracy']*100:.2f}% | F1-macro = {hang['F1-macro']*100:.2f}% | "
          f"Toxic Recall = {hang['Toxic Recall']*100:.2f}% | Toxic Precision = {hang['Toxic Precision']*100:.2f}%")

def trong_so_lop(y):
    dem = np.maximum(np.bincount(y, minlength=N_LOP), 1)
    return len(y) / (N_LOP * dem)

### 🔹 Mô hình 1 — TF-IDF + Logistic Regression (mô hình nền / baseline)

TF-IDF biến mỗi bình luận thành một dãy số dựa trên tần suất từ (cả cụm 1–3 từ). Logistic Regression học từ đó. Mô hình này **rất nhanh và nhẹ**, dùng làm mốc để so sánh. `class_weight="balanced"` giúp mô hình chú ý hơn tới nhóm ít mẫu (Toxic).

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

t0 = time.time()
tfidf = TfidfVectorizer(ngram_range=(1, 3), min_df=2, max_features=50000, sublinear_tf=True)
Xtr = tfidf.fit_transform(X_train)
lr = LogisticRegression(max_iter=1000, class_weight="balanced", random_state=SEED)
lr.fit(Xtr, y_train)
giay = time.time() - t0

def du_doan_tfidf(texts):
    proba = lr.predict_proba(tfidf.transform(texts))
    day_du = np.zeros((len(texts), N_LOP))
    day_du[:, lr.classes_] = proba      # phòng trường hợp thiếu nhãn nào đó
    return day_du

mb = len(pickle.dumps((tfidf, lr))) / 1e6
ghi_ket_qua("TF-IDF + Logistic Regression", du_doan_tfidf, giay, mb)

### 🔹 Mô hình 2 — BiLSTM (deep learning cổ điển)

BiLSTM đọc câu theo **cả hai chiều** nên hiểu được thứ tự từ. Ở đây mô hình tự học vector từ từ dữ liệu của nhóm (không dùng embedding có sẵn), nên với dữ liệu ít thì có thể không hơn TF-IDF nhiều. Đó là điều bình thường và cũng là một nhận xét hay cho báo cáo.

In [ ]:
if RUN_LSTM:
    import torch, torch.nn as nn
    from torch.utils.data import TensorDataset, DataLoader
    from collections import Counter
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    torch.manual_seed(SEED)

    dem_tu = Counter(t for s in X_train for t in s.split())
    vocab = {"<pad>": 0, "<unk>": 1}
    for tu, c in dem_tu.items():
        if c >= 2: vocab[tu] = len(vocab)

    def ma_hoa(texts):
        out = np.zeros((len(texts), MAX_LEN), dtype=np.int64)
        for i, s in enumerate(texts):
            ids = [vocab.get(t, 1) for t in s.split()[:MAX_LEN]]
            out[i, :len(ids)] = ids
        return torch.tensor(out)

    class BiLSTM(nn.Module):
        def __init__(self, v, n_lop, emb=128, hid=128):
            super().__init__()
            self.emb = nn.Embedding(v, emb, padding_idx=0)
            self.lstm = nn.LSTM(emb, hid, batch_first=True, bidirectional=True)
            self.drop = nn.Dropout(0.3)
            self.fc = nn.Linear(hid * 2, n_lop)
        def forward(self, x):
            mask = (x != 0).unsqueeze(-1)
            h, _ = self.lstm(self.emb(x))
            h = h.masked_fill(~mask, -1e9).max(dim=1).values
            return self.fc(self.drop(h))

    def proba_lstm_tensor(model, x_tensor, batch=256):
        model.eval(); ket_qua = []
        with torch.no_grad():
            for i in range(0, len(x_tensor), batch):
                lo = model(x_tensor[i:i + batch].to(DEVICE))
                ket_qua.append(torch.softmax(lo, dim=1).cpu().numpy())
        return np.vstack(ket_qua)

    xtr, xva = ma_hoa(X_train), ma_hoa(X_val)
    ytr = torch.tensor(y_train)
    model_lstm = BiLSTM(len(vocab), N_LOP).to(DEVICE)
    opt = torch.optim.Adam(model_lstm.parameters(), lr=1e-3)
    loss_fn = nn.CrossEntropyLoss(weight=torch.tensor(trong_so_lop(y_train), dtype=torch.float32).to(DEVICE))
    loader = DataLoader(TensorDataset(xtr, ytr), batch_size=64, shuffle=True)

    t0 = time.time(); tot_nhat, state_tot = -1, None
    for ep in range(1, LSTM_EPOCHS + 1):
        model_lstm.train()
        for xb, yb in loader:
            opt.zero_grad()
            loss = loss_fn(model_lstm(xb.to(DEVICE)), yb.to(DEVICE))
            loss.backward(); opt.step()
        f1_val = f1_score(y_val, proba_lstm_tensor(model_lstm, xva).argmax(1), average="macro")
        print(f"Epoch {ep}/{LSTM_EPOCHS} | loss lần cuối = {loss.item():.4f} | F1-macro trên val = {f1_val:.4f}")
        if f1_val > tot_nhat:
            tot_nhat, state_tot = f1_val, copy.deepcopy(model_lstm.state_dict())
    model_lstm.load_state_dict(state_tot)     # lấy lại epoch tốt nhất
    giay = time.time() - t0

    def du_doan_lstm(texts):
        return proba_lstm_tensor(model_lstm, ma_hoa(texts))

    mb = sum(p.numel() for p in model_lstm.parameters()) * 4 / 1e6
    ghi_ket_qua("BiLSTM", du_doan_lstm, giay, mb)
else:
    print("Bỏ qua BiLSTM (RUN_LSTM = False)")

### 🔹 Mô hình 3 — PhoBERT (fine-tune)

PhoBERT là mô hình ngôn ngữ **đã được huấn luyện sẵn trên rất nhiều văn bản tiếng Việt**. Ta chỉ cần "tinh chỉnh" (fine-tune) nó cho bài toán phân loại bình luận độc hại. Đây thường là mô hình mạnh nhất, nhưng cũng **nặng và chậm hơn** (xem cột kích thước và độ trễ trong bảng so sánh).

Lưu ý: cần bật **GPU** trên Colab, nếu không sẽ rất chậm.

In [ ]:
if RUN_PHOBERT:
    import torch, torch.nn as nn
    from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    if DEVICE.type == "cpu":
        print("⚠️ Đang chạy bằng CPU, PhoBERT sẽ rất chậm. Hãy bật GPU: Runtime → Change runtime type.")
    torch.manual_seed(SEED)

    tok = AutoTokenizer.from_pretrained(PHOBERT_NAME)
    model_pho = AutoModelForSequenceClassification.from_pretrained(PHOBERT_NAME, num_labels=N_LOP).to(DEVICE)

    def ma_hoa_pho(texts):
        enc = tok(texts, padding=True, truncation=True, max_length=MAX_LEN, return_tensors="pt")
        return enc["input_ids"], enc["attention_mask"]

    def proba_pho(texts, batch=64):
        model_pho.eval(); ket_qua = []
        with torch.no_grad():
            for i in range(0, len(texts), batch):
                ids, mask = ma_hoa_pho(texts[i:i + batch])
                lo = model_pho(input_ids=ids.to(DEVICE), attention_mask=mask.to(DEVICE)).logits
                ket_qua.append(torch.softmax(lo.float(), dim=1).cpu().numpy())
        return np.vstack(ket_qua)

    opt = torch.optim.AdamW(model_pho.parameters(), lr=2e-5, weight_decay=0.01)
    so_buoc = (len(X_train) // PHOBERT_BATCH + 1) * PHOBERT_EPOCHS
    sched = get_linear_schedule_with_warmup(opt, int(0.1 * so_buoc), so_buoc)
    loss_fn = nn.CrossEntropyLoss(weight=torch.tensor(trong_so_lop(y_train), dtype=torch.float32).to(DEVICE))
    scaler = torch.cuda.amp.GradScaler(enabled=(DEVICE.type == "cuda"))

    t0 = time.time(); tot_nhat, state_tot = -1, None
    for ep in range(1, PHOBERT_EPOCHS + 1):
        model_pho.train()
        thu_tu = np.random.permutation(len(X_train))
        for i in range(0, len(thu_tu), PHOBERT_BATCH):
            idx = thu_tu[i:i + PHOBERT_BATCH]
            ids, mask = ma_hoa_pho([X_train[j] for j in idx])
            yb = torch.tensor(y_train[idx]).to(DEVICE)
            opt.zero_grad()
            with torch.cuda.amp.autocast(enabled=(DEVICE.type == "cuda")):
                lo = model_pho(input_ids=ids.to(DEVICE), attention_mask=mask.to(DEVICE)).logits
                loss = loss_fn(lo.float(), yb)
            scaler.scale(loss).backward()
            scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(model_pho.parameters(), 1.0)
            scaler.step(opt); scaler.update(); sched.step()
        f1_val = f1_score(y_val, proba_pho(X_val).argmax(1), average="macro")
        print(f"Epoch {ep}/{PHOBERT_EPOCHS} | loss lần cuối = {loss.item():.4f} | F1-macro trên val = {f1_val:.4f}")
        if f1_val > tot_nhat:
            tot_nhat = f1_val
            state_tot = {k: v.detach().cpu().clone() for k, v in model_pho.state_dict().items()}
    model_pho.load_state_dict(state_tot)      # lấy lại epoch tốt nhất
    giay = time.time() - t0

    mb = sum(p.numel() for p in model_pho.parameters()) * 4 / 1e6
    ghi_ket_qua("PhoBERT (fine-tune)", proba_pho, giay, mb)
else:
    print("Bỏ qua PhoBERT (RUN_PHOBERT = False)")

## 5️⃣ Bảng so sánh các mô hình

Đây là bảng chính cho báo cáo: **mô hình nào tốt nhất, đánh đổi gì** (độ chính xác ↔ tốc độ ↔ dung lượng). Cách đọc gợi ý:
- Nếu PhoBERT có F1-macro cao nhất nhưng độ trễ và dung lượng lớn → nói rõ đây là sự đánh đổi.
- Nếu hai mô hình gần bằng nhau → TF-IDF là lựa chọn rẻ và nhanh.
- Với bài toán kiểm duyệt, **Toxic Recall** cao nghĩa là ít bỏ sót bình luận độc hại.

In [ ]:
df_ss = pd.DataFrame(KET_QUA)
df_hien = df_ss.copy()
for cot in ["Accuracy", "F1-macro", "Toxic Precision", "Toxic Recall"]:
    df_hien[cot] = (df_hien[cot] * 100).round(2)
df_hien["Độ trễ (ms/bình luận)"] = df_hien["Độ trễ (ms/bình luận)"].round(2)
df_hien["Kích thước (MB)"] = df_hien["Kích thước (MB)"].round(1)
df_hien["Thời gian huấn luyện (giây)"] = df_hien["Thời gian huấn luyện (giây)"].round(1)
df_hien.rename(columns={"Accuracy": "Accuracy (%)", "F1-macro": "F1-macro (%)",
                        "Toxic Precision": "Toxic Precision (%)", "Toxic Recall": "Toxic Recall (%)"}, inplace=True)
if USE_DEMO_DATA:
    print("⚠️ BẢNG DƯỚI ĐÂY LÀ TỪ DỮ LIỆU MẪU, KHÔNG DÙNG ĐỂ BÁO CÁO")
display(df_hien)
df_hien.to_csv("outputs/bang_so_sanh_mo_hinh.csv", index=False)

chi_so = ["Accuracy", "F1-macro", "Toxic Precision", "Toxic Recall"]
dai = df_ss.melt(id_vars="Mô hình", value_vars=chi_so, var_name="Chỉ số", value_name="Giá trị")
dai["Giá trị"] = dai["Giá trị"] * 100
plt.figure(figsize=(9, 4.2))
sns.barplot(data=dai, x="Chỉ số", y="Giá trị", hue="Mô hình", palette="Set2")
plt.ylim(0, 105); plt.ylabel("%"); plt.xlabel(""); plt.title("So sánh các mô hình trên tập test")
plt.legend(loc="lower right", fontsize=8); plt.tight_layout()
plt.savefig("outputs/so_sanh_mo_hinh.png", dpi=150); plt.show()

## 6️⃣ Ma trận nhầm lẫn

- **Hàng** = nhãn thật, **cột** = mô hình đoán. Ô chéo chính (trên-trái xuống dưới-phải) là đoán đúng.
- **Bên trái**: số bình luận. **Bên phải**: % theo từng hàng (cho biết mỗi nhóm thật được đoán đúng bao nhiêu %, chính là **Recall** của nhóm đó).
- Chú ý ô "thật là Toxic nhưng đoán Bình thường" → đây là các bình luận **bị bỏ sót**.

In [ ]:
nhan_ve = list(range(N_LOP))
for ten, proba in DU_DOAN.items():
    cm = confusion_matrix(y_test, proba.argmax(1), labels=nhan_ve)
    cm_pt = cm / np.maximum(cm.sum(axis=1, keepdims=True), 1) * 100
    fig, ax = plt.subplots(1, 2, figsize=(11, 4))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=LABEL_NAMES, yticklabels=LABEL_NAMES, ax=ax[0])
    ax[0].set_title(f"{ten}: số lượng"); ax[0].set_xlabel("Mô hình đoán"); ax[0].set_ylabel("Nhãn thật")
    sns.heatmap(cm_pt, annot=True, fmt=".1f", cmap="Oranges", xticklabels=LABEL_NAMES, yticklabels=LABEL_NAMES, ax=ax[1])
    ax[1].set_title(f"{ten}: % theo hàng"); ax[1].set_xlabel("Mô hình đoán"); ax[1].set_ylabel("Nhãn thật")
    plt.tight_layout()
    plt.savefig(f"outputs/ma_tran_nham_lan_{re.sub(r'[^A-Za-z0-9]+', '_', ten)}.png", dpi=150); plt.show()

## 7️⃣ Phân tích lỗi: mô hình đoán sai ở đâu và vì sao?

Lấy **mô hình có F1-macro cao nhất** rồi tách 2 loại lỗi (nhìn theo nhóm Toxic / không Toxic):
- **False Positive (FP)**: câu bình thường nhưng bị báo là độc hại → có thể **chặn nhầm** người dùng.
- **False Negative (FN)**: câu độc hại nhưng bị bỏ sót → **lọt lưới**.

Cột **"Gợi ý nguyên nhân"** do code tự đoán dựa trên dấu hiệu đơn giản (teencode, câu quá ngắn, có emoji cười...). Đây chỉ là **gợi ý ban đầu**. Bạn cần **đọc từng câu và viết nhận xét của nhóm** vào cột trống `nhận_xét_của_nhóm` trong file CSV thì phần này mới thuyết phục.

In [ ]:
ten_tot = df_ss.sort_values("F1-macro", ascending=False).iloc[0]["Mô hình"]
proba_tot = DU_DOAN[ten_tot]
print(f"Phân tích lỗi cho mô hình tốt nhất: {ten_tot}")

loi = test.copy()
loi["nhan_that"] = [LABEL_NAMES[i] for i in y_test]
loi["nhan_doan"] = [LABEL_NAMES[i] for i in proba_tot.argmax(1)]
loi["xac_suat_doan"] = proba_tot.max(1).round(3)
loi["xac_suat_toxic"] = proba_tot[:, TOXIC_IDS].sum(1).round(3)
that_toxic = np.isin(y_test, TOXIC_IDS); doan_toxic = np.isin(proba_tot.argmax(1), TOXIC_IDS)
loi["loai_loi"] = np.select([(~that_toxic) & doan_toxic, that_toxic & (~doan_toxic)], ["FP", "FN"], default="")
dh = loi["text"].apply(phat_hien).apply(pd.Series)
loi = pd.concat([loi, dh], axis=1)
loi["so_tu"] = loi["text_seg"].str.split().str.len()

TU_PHU_DINH = {"không", "chẳng", "chưa", "đâu", "chả"}
TU_DUA = ["haha", "hihi", "kkk", "=))", "😂", "🤣", "😅", "lol"]

def goi_y_nguyen_nhan(r):
    g = []
    if r["so_chen_ky_tu"] > 0: g.append("chèn ký tự để né bộ lọc (khó cho tokenizer)")
    if r["so_teencode"] > 0: g.append("có teencode/viết tắt")
    if r["so_keo_dai"] > 0: g.append("chữ kéo dài")
    if r["so_url"] > 0: g.append("có link")
    if r["co_emoji"]: g.append("có emoji")
    if r["so_tu"] <= 5: g.append("câu rất ngắn, thiếu ngữ cảnh")
    if TU_PHU_DINH & set(r["text_clean"].split()): g.append("có từ phủ định (dễ hiểu sai nghĩa)")
    if r["loai_loi"] == "FP" and any(x in r["text"].lower() for x in TU_DUA):
        g.append("có thể là đùa vui / mỉa mai")
    if not g:
        g.append("không thấy dấu hiệu rõ: có thể thiếu ngữ cảnh hoặc nhãn gốc chưa chắc chắn")
    return "; ".join(g)

loi["goi_y_nguyen_nhan"] = loi.apply(goi_y_nguyen_nhan, axis=1)
n_fp, n_fn = (loi["loai_loi"] == "FP").sum(), (loi["loai_loi"] == "FN").sum()
n_khong_toxic, n_toxic = (~that_toxic).sum(), that_toxic.sum()
nham_muc = ((loi["nhan_that"] != loi["nhan_doan"]) & (loi["loai_loi"] == "")).sum()
print(f"FP (chặn nhầm) = {n_fp} / {n_khong_toxic} câu bình thường = {n_fp/max(n_khong_toxic,1)*100:.2f}% | "
      f"FN (bỏ sót) = {n_fn} / {n_toxic} câu độc hại = {n_fn/max(n_toxic,1)*100:.2f}% | "
      f"Nhầm giữa các mức độc hại (cùng nhóm Toxic) = {nham_muc}")

cot_hien = ["text", "nhan_that", "nhan_doan", "xac_suat_toxic", "goi_y_nguyen_nhan"]
fp_top = loi[loi["loai_loi"] == "FP"].sort_values("xac_suat_toxic", ascending=False).head(5)
fn_top = loi[loi["loai_loi"] == "FN"].sort_values("xac_suat_toxic", ascending=True).head(5)
print("\n🟠 FALSE POSITIVE — câu bình thường bị báo là độc hại (tự tin nhất trước):")
display(fp_top[cot_hien] if len(fp_top) else "Không có lỗi FP nào.")
print("🔴 FALSE NEGATIVE — câu độc hại bị bỏ sót (tự tin sai nhất trước):")
display(fn_top[cot_hien] if len(fn_top) else "Không có lỗi FN nào.")

xuat = pd.concat([loi[loi["loai_loi"] == "FP"].sort_values("xac_suat_toxic", ascending=False).head(20),
                  loi[loi["loai_loi"] == "FN"].sort_values("xac_suat_toxic").head(20)])[["loai_loi"] + cot_hien].copy()
xuat["nhận_xét_của_nhóm"] = ""
xuat.to_csv("outputs/phan_tich_loi.csv", index=False)
print("Đã lưu outputs/phan_tich_loi.csv — hãy mở file, đọc từng câu và điền cột 'nhận_xét_của_nhóm'.")

## 8️⃣ Nhập câu → ra kết quả (giống trang demo)

Hàm `phan_loai()` thực hiện đủ 3 bước: **làm sạch → tách từ → mô hình dự đoán**, rồi in ra nhãn, độ tin cậy và điểm độc hại. Hàm này cũng chính là "lõi" để gắn vào trang web (Streamlit/Gradio) hoặc backend của giao diện bạn thiết kế.

In [ ]:
def phan_loai(binh_luan, ten_mo_hinh=None, in_ket_qua=True):
    ten = ten_mo_hinh or ten_tot
    sach = lam_sach(binh_luan)
    seg = tach_tu(sach)
    p = HAM_DU_DOAN[ten]([seg])[0]
    k = int(p.argmax())
    kq = {"binh_luan": binh_luan, "sau_xu_ly": seg, "nhan": LABEL_NAMES[k], "do_tin_cay": float(p[k]),
          "diem_toxic": float(p[TOXIC_IDS].sum()), "mo_hinh": ten,
          "xac_suat_tung_nhan": {LABEL_NAMES[i]: round(float(p[i]), 4) for i in range(N_LOP)}}
    if in_ket_qua:
        print(f"Bình luận: {binh_luan} | Sau xử lý: {seg} | Kết quả: {kq['nhan']} ({kq['do_tin_cay']*100:.1f}%) | "
              f"Điểm Toxic: {kq['diem_toxic']*100:.1f}% | Mô hình: {ten}")
    return kq

for cau in ["Video hay quá, cảm ơn bạn nhiều nha!",
            "thằng này ngu như bò, câm miệng đi",
            "hàng như l** ý mn đừng mua lừa đảo đấy k dùng dc đâu https://shopee.vn/fake-shop"]:
    phan_loai(cau)

# 👉 Thử câu của riêng bạn ở đây:
cau_cua_toi = "Bạn ơi cho mình hỏi sản phẩm này còn hàng không?"
_ = phan_loai(cau_cua_toi)

## 9️⃣ Mở rộng: lấy bình luận YouTube và phân loại hàng loạt

Ý tưởng: dán link một video **công khai** → lấy bình luận bằng **YouTube Data API v3** → phân loại → thống kê tỷ lệ.

**Cách lấy API key (miễn phí):** vào Google Cloud Console → tạo project → bật *YouTube Data API v3* → *Credentials → Create API key*.

Lưu ý quan trọng:
- **Không dán API key vào code hay đưa lên GitHub.** Code dưới đây hỏi key bằng `getpass` (ô nhập ẩn).
- API có **hạn mức dùng mỗi ngày** (quota). Bạn nên xem hạn mức hiện hành trên trang chính thức của Google vì có thể thay đổi.
- Video có thể **tắt bình luận** (API sẽ báo lỗi), đó là bình thường.
- Chỉ giữ **nội dung bình luận, số like, thời gian**. **Không lưu tên người dùng** để bảo vệ quyền riêng tư.
- Chỉ dùng cho mục đích học tập và tuân thủ điều khoản của YouTube.

In [ ]:
import requests, getpass

def lay_video_id(link_hoac_id):
    m = re.search(r"(?:v=|youtu\.be/|shorts/|embed/)([A-Za-z0-9_-]{11})", link_hoac_id)
    if m: return m.group(1)
    if re.fullmatch(r"[A-Za-z0-9_-]{11}", link_hoac_id.strip()): return link_hoac_id.strip()
    raise ValueError("Không nhận ra video id. Hãy dán link dạng https://www.youtube.com/watch?v=XXXXXXXXXXX")

def lay_binh_luan_youtube(link, api_key, toi_da=300, http_get=requests.get):
    """Lấy tối đa `toi_da` bình luận gốc của 1 video công khai. Chỉ giữ nội dung, like, thời gian."""
    vid = lay_video_id(link)
    url = "https://www.googleapis.com/youtube/v3/commentThreads"
    dong, token = [], None
    while len(dong) < toi_da:
        tham_so = {"part": "snippet", "videoId": vid, "maxResults": min(100, toi_da - len(dong)),
                   "textFormat": "plainText", "order": "relevance", "key": api_key}
        if token: tham_so["pageToken"] = token
        r = http_get(url, params=tham_so, timeout=30)
        if r.status_code != 200:
            try: ly_do = r.json()["error"]["errors"][0]["reason"]
            except Exception: ly_do = r.text[:200]
            raise RuntimeError(f"YouTube API lỗi {r.status_code}: {ly_do} "
                               "(commentsDisabled = video tắt bình luận; quotaExceeded = hết hạn mức; keyInvalid = sai key)")
        du_lieu = r.json()
        for it in du_lieu.get("items", []):
            s = it["snippet"]["topLevelComment"]["snippet"]
            dong.append({"text": s.get("textDisplay", ""), "like": s.get("likeCount", 0), "thoi_gian": s.get("publishedAt", "")})
        token = du_lieu.get("nextPageToken")
        if not token: break
    return pd.DataFrame(dong[:toi_da])

def phan_loai_hang_loat(df_bl, ten_mo_hinh=None):
    ten = ten_mo_hinh or ten_tot
    kq = df_bl.copy()
    kq["text_seg"] = kq["text"].apply(lambda s: tach_tu(lam_sach(s)))
    kq = kq[kq["text_seg"].str.len() > 0].copy().reset_index(drop=True)
    p = HAM_DU_DOAN[ten](kq["text_seg"].tolist())
    kq["nhan"] = [LABEL_NAMES[i] for i in p.argmax(1)]
    kq["diem_toxic"] = p[:, TOXIC_IDS].sum(1).round(4)
    return kq

def bao_cao_youtube(link, api_key, toi_da=300):
    bl = lay_binh_luan_youtube(link, api_key, toi_da)
    kq = phan_loai_hang_loat(bl)
    kq.to_csv("outputs/ket_qua_youtube.csv", index=False)
    ty_le = kq["nhan"].value_counts(normalize=True).mul(100).round(2).rename("Tỷ lệ %").to_frame()
    print(f"Đã phân loại {len(kq)} bình luận từ video. Đã lưu outputs/ket_qua_youtube.csv")
    display(ty_le)
    print("Top 5 bình luận có điểm độc hại cao nhất:")
    display(kq.sort_values("diem_toxic", ascending=False).head(5)[["text", "nhan", "diem_toxic", "like"]])
    ty_le["Tỷ lệ %"].plot(kind="bar", color="#e45756", figsize=(6, 3.5))
    plt.ylabel("%"); plt.title("Tỷ lệ các nhóm bình luận"); plt.xticks(rotation=0); plt.tight_layout(); plt.show()
    return kq

In [ ]:
# 👉 Chạy phần này khi đã có API key và link video công khai. Để trống link = bỏ qua.
LINK_VIDEO = ""      # ví dụ: "https://www.youtube.com/watch?v=XXXXXXXXXXX"
if LINK_VIDEO.strip():
    API_KEY = getpass.getpass("Nhập YouTube API key (ô ẩn, không lưu vào file): ")
    df_youtube = bao_cao_youtube(LINK_VIDEO, API_KEY, toi_da=300)
else:
    print("Chưa nhập LINK_VIDEO nên bỏ qua phần YouTube. Điền link video công khai rồi chạy lại ô này.")

## 🔟 Lưu mô hình để dùng cho trang web demo

Ô này lưu mô hình đã huấn luyện vào thư mục `models/` và ghi file `models/config.json` (tên nhãn, mô hình tốt nhất...). Trang web (xem file bản vẽ giao diện) sẽ đọc các file này, nên **không cần huấn luyện lại**.

Sau khi chạy xong, **tải 2 thư mục `models/` và `outputs/` về máy** (trên Colab: bấm chuột phải vào thư mục → Download, hoặc dùng ô nén zip bên dưới).

In [ ]:
import json, shutil
os.makedirs("models", exist_ok=True)

with open("models/tfidf_lr.pkl", "wb") as f:
    pickle.dump((tfidf, lr), f)
if RUN_LSTM:
    torch.save({"state": model_lstm.state_dict(), "vocab": vocab}, "models/bilstm.pt")
if RUN_PHOBERT:
    model_pho.save_pretrained("models/phobert"); tok.save_pretrained("models/phobert")

cau_hinh = {"label_names": LABEL_NAMES, "toxic_ids": TOXIC_IDS, "best_model": ten_tot,
            "max_len": MAX_LEN, "models_available": list(HAM_DU_DOAN.keys()),
            "is_demo_data": bool(USE_DEMO_DATA)}
with open("models/config.json", "w", encoding="utf-8") as f:
    json.dump(cau_hinh, f, ensure_ascii=False, indent=2)
print("Đã lưu thư mục models/ gồm:", sorted(os.listdir("models")))

# Nén thành file zip để tải về cho nhanh
shutil.make_archive("ket_qua_outputs", "zip", root_dir=".", base_dir="outputs")
shutil.make_archive("ket_qua_models", "zip", root_dir=".", base_dir="models")
print("Đã tạo ket_qua_outputs.zip và ket_qua_models.zip — tải về máy ở khung Files bên trái.")

## ✅ Các file kết quả (thư mục `outputs/`) để đưa vào báo cáo / slide

| File | Dùng cho |
|---|---|
| `bang_so_sanh_mo_hinh.csv`, `so_sanh_mo_hinh.png` | Bảng và biểu đồ so sánh 3 mô hình |
| `ma_tran_nham_lan_*.png` | Ma trận nhầm lẫn từng mô hình |
| `phan_tich_loi.csv` | Phân tích lỗi (điền thêm nhận xét của nhóm) |
| `truoc_sau_xu_ly.csv`, `thong_ke_dau_hieu_ban.png` | Khung so sánh trước/sau xử lý |
| `phan_bo_nhan.png` | Phân bố nhãn |
| `ket_qua_youtube.csv` | Kết quả phân loại bình luận YouTube (nếu chạy) |
| `models/` (thư mục riêng) | Mô hình đã lưu + `config.json` cho trang web demo |

**Gợi ý nói khi bảo vệ:** (1) dữ liệu lấy ở đâu, ai gán nhãn; (2) vì sao chọn F1-macro; (3) mô hình nào tốt nhất và đánh đổi gì; (4) vài câu đoán sai và lý do; (5) hạn chế và vấn đề đạo đức (chặn nhầm, mỉa mai, ngữ cảnh).